In [2]:
import pandas as pd
import numpy as np


## Load Data

In [3]:
data = pd.read_csv('../data/external/env_wasgen_linear_2_0.csv')

In [ ]:
#Loading waste category excel to aggregate the waste categories in the data
categories = pd.read_excel('../data/processed/categories_msw_composition.xlsx')
categories = categories[['Aggregated categories', 'Aggregated category name']]
categories = categories.loc[categories['Aggregated categories'].notna()].reset_index()

Choosing household waste data

In [64]:
hhw = data.loc[data['Statistical classification of economic activities in the European Community (NACE Rev. 2)'] == 'Households'].copy().reset_index()

In [65]:
hhw=hhw[['unit', 'Unit of measure', 'hazard', 'Hazard class', 
       'waste', 'Waste categories', 'geo', 'Geopolitical entity (reporting)',
       'TIME_PERIOD',  'OBS_VALUE', 'OBS_FLAG',
       'Observation status (Flag) V2 structure']]

In [67]:
hhw = hhw.loc[hhw['waste'].isin(categories['Aggregated categories'].unique())].copy().reset_index(drop=True)

In [ ]:
#Getting the volume data (in tonnes)
hhw_t = hhw.loc[hhw['unit']=='T'].copy().reset_index(drop=True)

Summing WEEE and battery waste categories

In [71]:
weee_batt_cat = ['W077', 'W0841', 'W08A']
weee = hhw_t.loc[hhw_t['waste'].isin(weee_batt_cat)].copy().reset_index(drop=True)


In [ ]:
weee = weee.groupby(['unit', 'Unit of measure', 'geo', 'Geopolitical entity (reporting)',
       'TIME_PERIOD'], as_index=False)['OBS_VALUE'].sum()


In [78]:
weee['waste'] = 'W07_08'
weee['Waste categories'] = 'weee + batt, no elv'

In [79]:
hhw_t = hhw_t.loc[~hhw_t['waste'].isin(weee_batt_cat)].copy()
hhw_t = pd.concat([hhw_t, weee], ignore_index=True)

In [85]:
hhw_t.columns

Index(['unit', 'Unit of measure', 'hazard', 'Hazard class', 'waste',
       'Waste categories', 'geo', 'Geopolitical entity (reporting)',
       'TIME_PERIOD', 'OBS_VALUE', 'OBS_FLAG',
       'Observation status (Flag) V2 structure'],
      dtype='object')

In [81]:
hhw_t['Waste categories'].unique()

array(['Health care and biological wastes', 'Metal wastes, ferrous',
       'Metal wastes, non-ferrous',
       'Metal wastes, mixed ferrous and non-ferrous', 'Glass wastes',
       'Paper and cardboard wastes', 'Rubber wastes', 'Plastic wastes',
       'Wood wastes', 'Textile wastes',
       'Mixed ordinary wastes (subtotal, W101+W102+W103)',
       'Animal and mixed food waste; vegetal wastes (W091+W092)',
       'weee + batt, no elv'], dtype=object)

In [86]:
hhw_t.drop(columns = ['Unit of measure','hazard', 'Hazard class', 'OBS_FLAG', 'Observation status (Flag) V2 structure'], inplace=True)

In [98]:
hhw_t = hhw_t.pivot_table(index = ['unit','geo', 'Geopolitical entity (reporting)',
       'TIME_PERIOD'], columns = 'Waste categories', values='OBS_VALUE').reset_index()

In [99]:
id_cols = ['unit','geo','Geopolitical entity (reporting)','TIME_PERIOD']
cat_cols = hhw_t.columns.difference(id_cols)

In [100]:
hhw_percent = hhw_t.copy()
hhw_percent[cat_cols] = hhw_t[cat_cols].div(
    hhw_t[cat_cols].sum(axis=1).replace(0, np.nan), axis=0
) * 100

In [102]:
hhw_percent[hhw_percent['Geopolitical entity (reporting)']=='Austria']

Waste categories,unit,geo,Geopolitical entity (reporting),TIME_PERIOD,Animal and mixed food waste; vegetal wastes (W091+W092),Glass wastes,Health care and biological wastes,"Metal wastes, ferrous","Metal wastes, mixed ferrous and non-ferrous","Metal wastes, non-ferrous","Mixed ordinary wastes (subtotal, W101+W102+W103)",Paper and cardboard wastes,Plastic wastes,Rubber wastes,Textile wastes,Wood wastes,"weee + batt, no elv"
7,T,AT,Austria,2004,20.564818,4.662864,0.000000,NaN,NaN,NaN,40.936247,22.627699,4.780765,0.000000,0.865808,3.044256,2.517543
8,T,AT,Austria,2006,22.443384,4.421027,0.000000,NaN,NaN,NaN,38.195429,22.222785,4.384826,0.000000,0.753430,3.234544,4.344575
9,T,AT,Austria,2008,23.381794,4.545337,0.000000,NaN,NaN,NaN,35.830608,23.018230,4.644732,0.000000,0.891184,3.551935,4.136180
10,T,AT,Austria,2010,NaN,10.577666,0.000522,6.674579,0.0,1.684041,NaN,48.200903,10.783332,0.245775,1.851143,9.360326,10.621713
11,T,AT,Austria,2012,NaN,10.597278,0.000479,8.136068,0.0,0.284751,NaN,45.882388,10.968779,0.000000,1.970625,10.580896,11.578735
12,T,AT,Austria,2014,NaN,10.391380,0.000640,7.763170,0.0,0.122765,NaN,46.498457,11.270523,0.245668,1.904131,10.713493,11.089772
13,T,AT,Austria,2016,NaN,10.099958,0.000406,8.202780,0.0,0.110120,NaN,44.347665,11.423042,0.282470,2.296419,11.452015,11.785126
14,T,AT,Austria,2018,NaN,9.965691,0.000422,7.765304,0.0,0.107955,NaN,41.420484,11.177195,0.062620,2.386222,11.975177,15.138930
15,T,AT,Austria,2020,NaN,10.548936,0.000426,8.465486,0.0,0.175177,NaN,38.006888,11.309314,0.000000,2.331435,11.879673,17.282665
16,T,AT,Austria,2022,NaN,11.276395,0.013939,7.155570,0.0,0.173606,NaN,38.528141,11.872137,0.000000,2.343075,10.711326,17.925811


In [ ]:
hhw_percent

array([2022, 2010, 2012, 2014, 2016, 2018, 2020, 2004, 2006, 2008])

In [111]:
data.loc[(data['Geopolitical entity (reporting)']=='Austria')&(data['Statistical classification of economic activities in the European Community (NACE Rev. 2)']=='Households')]

,STRUCTURE,STRUCTURE_ID,STRUCTURE_NAME,freq,Time frequency,unit,Unit of measure,hazard,Hazard class,nace_r2,...,geo,Geopolitical entity (reporting),TIME_PERIOD,Time,OBS_VALUE,Observation value,OBS_FLAG,Observation status (Flag) V2 structure,CONF_STATUS,Confidentiality status (flag)
206062,dataflow,ESTAT:ENV_WASGEN(1.0),"Generation of waste by waste category, hazardo...",A,Annual,KG_HAB,Kilograms per capita,HAZ,Hazardous,EP_HH,...,AT,Austria,2010,NaN,12.0,NaN,NaN,NaN,NaN,NaN
206063,dataflow,ESTAT:ENV_WASGEN(1.0),"Generation of waste by waste category, hazardo...",A,Annual,KG_HAB,Kilograms per capita,HAZ,Hazardous,EP_HH,...,AT,Austria,2012,NaN,9.0,NaN,NaN,NaN,NaN,NaN
206064,dataflow,ESTAT:ENV_WASGEN(1.0),"Generation of waste by waste category, hazardo...",A,Annual,KG_HAB,Kilograms per capita,HAZ,Hazardous,EP_HH,...,AT,Austria,2014,NaN,8.0,NaN,NaN,NaN,NaN,NaN
206065,dataflow,ESTAT:ENV_WASGEN(1.0),"Generation of waste by waste category, hazardo...",A,Annual,KG_HAB,Kilograms per capita,HAZ,Hazardous,EP_HH,...,AT,Austria,2016,NaN,9.0,NaN,NaN,NaN,NaN,NaN
206066,dataflow,ESTAT:ENV_WASGEN(1.0),"Generation of waste by waste category, hazardo...",A,Annual,KG_HAB,Kilograms per capita,HAZ,Hazardous,EP_HH,...,AT,Austria,2018,NaN,8.0,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1610679,dataflow,ESTAT:ENV_WASGEN(1.0),"Generation of waste by waste category, hazardo...",A,Annual,T,Tonne,NHAZ,Non-hazardous,EP_HH,...,AT,Austria,2020,NaN,0.0,NaN,NaN,NaN,NaN,NaN
1610680,dataflow,ESTAT:ENV_WASGEN(1.0),"Generation of waste by waste category, hazardo...",A,Annual,T,Tonne,NHAZ,Non-hazardous,EP_HH,...,AT,Austria,2022,NaN,0.0,NaN,NaN,NaN,NaN,NaN
1610915,dataflow,ESTAT:ENV_WASGEN(1.0),"Generation of waste by waste category, hazardo...",A,Annual,T,Tonne,NHAZ,Non-hazardous,EP_HH,...,AT,Austria,2004,NaN,0.0,NaN,NaN,NaN,NaN,NaN
1610916,dataflow,ESTAT:ENV_WASGEN(1.0),"Generation of waste by waste category, hazardo...",A,Annual,T,Tonne,NHAZ,Non-hazardous,EP_HH,...,AT,Austria,2006,NaN,0.0,NaN,NaN,NaN,NaN,NaN


In [5]:
data['Waste categories'].unique()

array(['Primary waste (TOTAL minus SEC)',
       'Secondary waste (W033+W103+W128_13)', 'Total waste',
       'Waste excluding major mineral wastes',
       'Chemical and medical wastes (subtotal)', 'Spent solvents',
       'Acid, alkaline or saline wastes', 'Used oils', 'Chemical wastes',
       'Industrial effluent sludges',
       'Sludges and liquid wastes from waste treatment',
       'Health care and biological wastes',
       'Metallic wastes (W061+W062+W063)', 'Metal wastes, ferrous',
       'Metal wastes, non-ferrous',
       'Metal wastes, mixed ferrous and non-ferrous',
       'Recyclable wastes (subtotal, W06+W07 except W077)',
       'Glass wastes', 'Paper and cardboard wastes', 'Rubber wastes',
       'Plastic wastes', 'Wood wastes', 'Textile wastes',
       'Waste containing PCB',
       'Equipment (subtotal, W077+W08A+W081+W0841)', 'Discarded vehicles',
       'Batteries and accumulators wastes',
       'Discarded equipment (except discarded vehicles and batteries and a